# 01 — College football game results

Pulls every FBS game result for the season from the [CollegeFootballData.com](https://collegefootballdata.com) API
into a pandas DataFrame (one row per game), builds team records, and joins in the CFP committee rankings.

**Setup:** put your free API key (from https://collegefootballdata.com/key) in a `.env` file at the repo root as `CFBD_API_KEY=...`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")  # repo-root .env when running from notebooks/
load_dotenv()                            # or a .env in the current directory

API_KEY = os.environ.get("CFBD_API_KEY")
if not API_KEY or API_KEY == "your_key_here":
    raise RuntimeError("Set CFBD_API_KEY in .env (get a free key at https://collegefootballdata.com/key)")

YEAR = 2026
BASE_URL = "https://api.collegefootballdata.com"
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Accept": "application/json"}

pd.set_option("display.max_columns", 50)

## Pull games

In [ ]:
def cfbd_get(endpoint, **params):
    resp = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


def get_games(year, season_type="regular"):
    return pd.DataFrame(cfbd_get("games", year=year, seasonType=season_type, classification="fbs"))


raw = pd.concat(
    [get_games(YEAR, "regular"), get_games(YEAR, "postseason")],
    ignore_index=True,
)
print(raw.shape)
raw.columns.tolist()

In [ ]:
COLUMNS = {
    "id": "game_id",
    "season": "season",
    "week": "week",
    "seasonType": "season_type",
    "startDate": "start_date",
    "neutralSite": "neutral_site",
    "conferenceGame": "conference_game",
    "homeTeam": "home_team",
    "homeConference": "home_conference",
    "homeClassification": "home_classification",
    "homePoints": "home_points",
    "awayTeam": "away_team",
    "awayConference": "away_conference",
    "awayClassification": "away_classification",
    "awayPoints": "away_points",
}

games = (
    raw[raw["completed"]]
    .loc[:, [c for c in COLUMNS if c in raw.columns]]
    .rename(columns=COLUMNS)
    .assign(start_date=lambda d: pd.to_datetime(d["start_date"], utc=True))
    .sort_values(["start_date", "game_id"])
    .reset_index(drop=True)
)

games["home_win"] = games["home_points"] > games["away_points"]
games["winner"] = games["home_team"].where(games["home_win"], games["away_team"])
games["loser"] = games["away_team"].where(games["home_win"], games["home_team"])
games["margin"] = (games["home_points"] - games["away_points"]).abs()

print(f"{len(games)} completed games")
games.head()

In [ ]:
games.groupby(["season_type", "week"]).size().rename("games")

## Team records

One row per team per game (long format), then aggregated. Only FBS teams are kept in the summary.

In [ ]:
def side(df, me, opp):
    return pd.DataFrame({
        "game_id": df["game_id"],
        "week": df["week"],
        "team": df[f"{me}_team"],
        "conference": df[f"{me}_conference"],
        "classification": df.get(f"{me}_classification"),
        "opponent": df[f"{opp}_team"],
        "points_for": df[f"{me}_points"],
        "points_against": df[f"{opp}_points"],
        "home": me == "home",
    })

team_games = pd.concat([side(games, "home", "away"), side(games, "away", "home")], ignore_index=True)
team_games["win"] = team_games["points_for"] > team_games["points_against"]
team_games["margin"] = team_games["points_for"] - team_games["points_against"]

records = (
    team_games[team_games["classification"].fillna("fbs") == "fbs"]
    .groupby(["team", "conference"])
    .agg(
        games=("win", "size"),
        wins=("win", "sum"),
        points_for=("points_for", "sum"),
        points_against=("points_against", "sum"),
        avg_margin=("margin", "mean"),
    )
    .assign(losses=lambda d: d["games"] - d["wins"])
    .reset_index()
    .sort_values(["wins", "avg_margin"], ascending=False)
)
records.head(25)

## CFP committee rankings

In [ ]:
def get_cfp_rankings(year):
    rows = []
    for week in cfbd_get("rankings", year=year):
        for poll in week["polls"]:
            if poll["poll"] != "Playoff Committee Rankings":
                continue
            for r in poll["ranks"]:
                rows.append({
                    "season_type": week["seasonType"],
                    "week": week["week"],
                    "rank": r["rank"],
                    "team": r["school"],
                    "conference": r["conference"],
                })
    return pd.DataFrame(rows)


cfp = get_cfp_rankings(YEAR)
if cfp.empty:
    print("No committee rankings released yet.")
else:
    # Wide view: one row per team, one column per ranking week
    print(cfp.pivot_table(index="team", columns="week", values="rank").sort_values(cfp["week"].max()).to_string())

In [ ]:
if not cfp.empty:
    latest = cfp[cfp["week"] == cfp["week"].max()][["team", "rank"]].rename(columns={"rank": "cfp_rank"})
    ranked_records = records.merge(latest, on="team", how="left").sort_values("cfp_rank")
    display(ranked_records.head(25))

## Save for later notebooks

In [ ]:
data_dir = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path("data")
data_dir.mkdir(exist_ok=True)
games.to_csv(data_dir / f"games_{YEAR}.csv", index=False)
team_games.to_csv(data_dir / f"team_games_{YEAR}.csv", index=False)
if not cfp.empty:
    cfp.to_csv(data_dir / f"cfp_rankings_{YEAR}.csv", index=False)
print("Saved to", data_dir.resolve())